## M2 — Variables de demanda
Calculamos las variables de negocio a partir de los datos limpios del M1.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Cargar los datasets limpios del M1
df_pir  = pd.read_csv('piramide_limpia.csv', dtype={'seccion_censal': str})
df_loc  = pd.read_csv('localizacion_limpia.csv')
df_muni = pd.read_csv('municipios_limpios.csv')
df_renta = pd.read_csv('renta_limpia.csv', encoding='utf-8-sig')

print("Datasets cargados:")
print(f"  Pirámide:      {df_pir.shape[0]:,} filas")
print(f"  Localización:  {df_loc.shape[0]:,} filas")
print(f"  Municipios:    {df_muni.shape[0]:,} filas")
print(f"  Renta:         {df_renta.shape[0]:,} filas")

Datasets cargados:
  Pirámide:      36,334 filas
  Localización:  36,333 filas
  Municipios:    8,124 filas
  Renta:         6,811 filas


### Paso 1 — Calcular volumen y porcentaje de mayores de 75+

In [2]:
# Grupos de edad que forman la población 75+
cols_75plus = [
    'ambos_75_79', 'ambos_80_84', 'ambos_85_89',
    'ambos_90_94', 'ambos_95_99', 'ambos_100mas'
]

# Volumen absoluto — cuántas personas de 75+ hay en la sección
df_pir['vol_75plus'] = df_pir[cols_75plus].sum(axis=1)

# Porcentaje sobre el total de la sección
df_pir['pct_75plus'] = df_pir['vol_75plus'] / df_pir['total_ambos'] * 100

print("Variables calculadas:")
print(f"  vol_75plus — min: {df_pir['vol_75plus'].min():.0f}, "
      f"max: {df_pir['vol_75plus'].max():.0f}, "
      f"media: {df_pir['vol_75plus'].mean():.1f}")
print(f"  pct_75plus — min: {df_pir['pct_75plus'].min():.1f}%, "
      f"max: {df_pir['pct_75plus'].max():.1f}%, "
      f"media: {df_pir['pct_75plus'].mean():.1f}%")

df_pir[['seccion_censal', 'total_ambos', 'vol_75plus', 'pct_75plus']].head(8)

Variables calculadas:
  vol_75plus — min: 0, max: 2505, media: 127.5
  pct_75plus — min: 0.0%, max: 61.5%, media: 11.8%


,seccion_censal,total_ambos,vol_75plus,pct_75plus
0,0100101001,1382.0,80.0,5.788712
1,0100101002,1543.0,88.0,5.703176
2,0100201001,1888.0,197.0,10.434322
3,0100201002,1569.0,167.0,10.643722
4,0100201003,1505.0,105.0,6.976744
5,0100201004,1815.0,164.0,9.035813
6,0100201005,1221.0,178.0,14.578215
7,0100201006,1368.0,94.0,6.871345


### Paso 2 — Calcular el ratio de viudez

In [3]:
# Mujeres 80+ y hombres 80+
cols_mujeres_80plus = [
    'mujeres_80_84', 'mujeres_85_89',
    'mujeres_90_94', 'mujeres_95_99', 'mujeres_100mas'
]
cols_hombres_80plus = [
    'hombres_80_84', 'hombres_85_89',
    'hombres_90_94', 'hombres_95_99', 'hombres_100mas'
]

df_pir['mujeres_80plus'] = df_pir[cols_mujeres_80plus].sum(axis=1)
df_pir['hombres_80plus'] = df_pir[cols_hombres_80plus].sum(axis=1)

# Ratio viudez — clip(lower=1) evita división por cero
df_pir['ratio_viudez'] = (
    df_pir['mujeres_80plus'] / df_pir['hombres_80plus'].clip(lower=1)
)

print("Ratio de viudez calculado:")
print(f"  min:    {df_pir['ratio_viudez'].min():.2f}")
print(f"  max:    {df_pir['ratio_viudez'].max():.2f}")
print(f"  media:  {df_pir['ratio_viudez'].mean():.2f}")
print(f"\nSecciones con ratio > 2.0 (alta viudez): "
      f"{(df_pir['ratio_viudez'] > 2.0).sum():,}")

df_pir[['seccion_censal', 'mujeres_80plus',
        'hombres_80plus', 'ratio_viudez']].head(8)

Ratio de viudez calculado:
  min:    0.00
  max:    25.00
  media:  1.78

Secciones con ratio > 2.0 (alta viudez): 9,661


,seccion_censal,mujeres_80plus,hombres_80plus,ratio_viudez
0,0100101001,21.0,20.0,1.050000
1,0100101002,32.0,18.0,1.777778
2,0100201001,80.0,49.0,1.632653
3,0100201002,56.0,31.0,1.806452
4,0100201003,31.0,23.0,1.347826
5,0100201004,71.0,33.0,2.151515
6,0100201005,74.0,33.0,2.242424
7,0100201006,36.0,19.0,1.894737


### Paso 3 — Calcular la cohorte futura

In [4]:
# Personas de 65-74 hoy → clientes potenciales en 2029
cols_cohorte = ['ambos_65_69', 'ambos_70_74']

df_pir['cohorte_futura'] = df_pir[cols_cohorte].sum(axis=1)

# También calculamos vol_65plus para tener una visión más amplia
cols_65plus = [
    'ambos_65_69', 'ambos_70_74', 'ambos_75_79', 'ambos_80_84',
    'ambos_85_89', 'ambos_90_94', 'ambos_95_99', 'ambos_100mas'
]
df_pir['vol_65plus'] = df_pir[cols_65plus].sum(axis=1)
df_pir['pct_65plus'] = df_pir['vol_65plus'] / df_pir['total_ambos'] * 100

print("Cohorte futura calculada:")
print(f"  cohorte_futura — min: {df_pir['cohorte_futura'].min():.0f}, "
      f"max: {df_pir['cohorte_futura'].max():.0f}, "
      f"media: {df_pir['cohorte_futura'].mean():.1f}")
print(f"\n  vol_65plus — media: {df_pir['vol_65plus'].mean():.1f}")
print(f"  pct_65plus — media: {df_pir['pct_65plus'].mean():.1f}%")

df_pir[['seccion_censal', 'total_ambos',
        'cohorte_futura', 'vol_65plus', 'pct_65plus']].head(8)

Cohorte futura calculada:
  cohorte_futura — min: 0, max: 3079, media: 128.8

  vol_65plus — media: 256.3
  pct_65plus — media: 22.5%


,seccion_censal,total_ambos,cohorte_futura,vol_65plus,pct_65plus
0,0100101001,1382.0,98.0,178.0,12.879884
1,0100101002,1543.0,98.0,186.0,12.054439
2,0100201001,1888.0,282.0,479.0,25.370763
3,0100201002,1569.0,263.0,430.0,27.405991
4,0100201003,1505.0,132.0,237.0,15.747508
5,0100201004,1815.0,192.0,356.0,19.614325
6,0100201005,1221.0,190.0,368.0,30.139230
7,0100201006,1368.0,140.0,234.0,17.105263


### Paso 4 — Cruzar pirámide con localización y renta

In [5]:
# Primero preparamos la columna de unión en localización
# seccion_censal en pirámide es string, en localización es int — los igualamos
df_loc['seccion_censal'] = df_loc['Seccion'].astype(str).str.zfill(10)

# Cruzar pirámide con localización
df = df_pir.merge(
    df_loc[['seccion_censal', 'longitud', 'latitud', 'CPRO', 'CMUN']],
    on='seccion_censal',
    how='left'
)

print(f"Tras cruzar con localización: {df.shape[0]:,} filas")
print(f"Secciones sin coordenadas: {df['latitud'].isnull().sum()}")

# Cruzar con renta por municipio (CPRO + CMUN)
df = df.merge(
    df_renta[['CPRO', 'CMUN', 'renta_persona_2024', 'renta_hogar_2024']],
    on=['CPRO', 'CMUN'],
    how='left'
)

print(f"Tras cruzar con renta:        {df.shape[0]:,} filas")
print(f"Secciones sin renta:          {df['renta_hogar_2024'].isnull().sum():,}")

df[['seccion_censal', 'latitud', 'longitud',
    'renta_hogar_2024', 'vol_75plus', 'pct_75plus']].head(5)

Tras cruzar con localización: 36,334 filas
Secciones sin coordenadas: 1
Tras cruzar con renta:        36,334 filas
Secciones sin renta:          1,322


,seccion_censal,latitud,longitud,renta_hogar_2024,vol_75plus,pct_75plus
0,0100101001,42.813781,-2.521144,34373.0,80.0,5.788712
1,0100101002,42.847201,-2.499572,34373.0,88.0,5.703176
2,0100201001,42.987151,-2.996599,33936.0,197.0,10.434322
3,0100201002,43.034617,-2.948063,33936.0,167.0,10.643722
4,0100201003,43.057505,-3.007986,33936.0,105.0,6.976744


### Paso 5 — Aplicar el filtro duro de solvencia

In [6]:
PRECIO_PLAZA_ANUAL = 2100 * 12  # 25.200€

# Marcar secciones solventes
df['solvente'] = (df['renta_hogar_2024'] >= PRECIO_PLAZA_ANUAL).astype(int)

# Resumen
total        = len(df)
solventes    = df['solvente'].sum()
no_solventes = total - solventes

print(f"Total secciones:          {total:,}")
print(f"Solventes (renta >= 25.200€): {solventes:,} "
      f"({solventes/total*100:.1f}%)")
print(f"Descartadas:              {no_solventes:,} "
      f"({no_solventes/total*100:.1f}%)")

# Ver la distribución de renta
print(f"\nRenta media hogar:")
print(f"  Secciones solventes:     "
      f"{df[df['solvente']==1]['renta_hogar_2024'].mean():,.0f}€")
print(f"  Secciones no solventes:  "
      f"{df[df['solvente']==0]['renta_hogar_2024'].mean():,.0f}€")

Total secciones:          36,334
Solventes (renta >= 25.200€): 24,438 (67.3%)
Descartadas:              11,896 (32.7%)

Renta media hogar:
  Secciones solventes:     32,229€
  Secciones no solventes:  21,658€


### Paso 6 — Guardar el dataset maestro

In [7]:
columnas_maestro = [
    'seccion_censal', 'CPRO', 'CMUN',
    'latitud', 'longitud',
    'total_ambos',
    'vol_65plus', 'pct_65plus',
    'vol_75plus', 'pct_75plus',
    'cohorte_futura',
    'mujeres_80plus', 'hombres_80plus', 'ratio_viudez',
    'renta_persona_2024', 'renta_hogar_2024',
    'solvente'
]

df_maestro = df[columnas_maestro].copy()

# Resumen final
print("=== DATASET MAESTRO ===")
print(f"Filas:    {df_maestro.shape[0]:,}")
print(f"Columnas: {df_maestro.shape[1]}")
print(f"\nNulos por columna:")
print(df_maestro.isnull().sum())

df_maestro.to_csv('dataset_maestro.csv', index=False, encoding='utf-8-sig')
print("\ndataset_maestro.csv guardado")

=== DATASET MAESTRO ===
Filas:    36,334
Columnas: 17

Nulos por columna:
seccion_censal           0
CPRO                     1
CMUN                     1
latitud                  1
longitud                 1
total_ambos              0
vol_65plus               0
pct_65plus               0
vol_75plus               0
pct_75plus               0
cohorte_futura           0
mujeres_80plus           0
hombres_80plus           0
ratio_viudez             0
renta_persona_2024    1322
renta_hogar_2024      1322
solvente                 0
dtype: int64

dataset_maestro.csv guardado
